# STEP1：成人の初回ICU入室を抽出する

MIMIC-IV 3.1の`hosp/patients.csv.gz`と`icu/icustays.csv.gz`を直接読み込む。
患者ごとにICU入室時刻が最も早い1件を採用し、その時点の年齢が18歳以上の患者を`df_step1`に抽出する。目標の確認値は **N = 65,366人**。

複数回のICU入室記録がある患者も、最初の1件を採用する。ここでは診断、ICU滞在時間、臨床データの欠測による選択条件は適用しない。

In [41]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# ノートブックのディレクトリ（original/）から実行する。
DATA_DIR = Path("3.1").resolve()
print(f"読み込み元: {DATA_DIR}")

patients = pd.read_csv(DATA_DIR / "hosp/patients.csv.gz")
icustays = pd.read_csv(DATA_DIR / "icu/icustays.csv.gz",
                       parse_dates=["intime", "outtime"])
assert patients.subject_id.is_unique
assert icustays.stay_id.is_unique
assert icustays.intime.notna().all(), "初回入室を決めるためのintimeが欠測しています。"
print(f"全ICU入室記録: {len(icustays):,}件")
print(f"ICU入室歴のある患者: {icustays.subject_id.nunique():,}人")

読み込み元: /Users/uminchu/GitHub/admm/data/real/MIMIC-IV/original/3.1
全ICU入室記録: 94,458件
ICU入室歴のある患者: 65,366人


## 患者ごとの初回入室と成人判定

`subject_id`ごとに`intime`の昇順で並べ、最初のICU入室を選ぶ。同じ時刻の記録があれば`stay_id`の昇順で決める。
年齢は`anchor_age + ICU入室年 - anchor_year`として算出する。年齢にはMIMIC-IVの匿名化処理が反映されている。

In [42]:
icu_stay_count = icustays.groupby("subject_id").size().rename("icu_stay_count")
first_icu = (
    icustays.sort_values(["subject_id", "intime", "stay_id"], kind="stable")
    .drop_duplicates("subject_id", keep="first")
    .merge(patients, on="subject_id", how="left", validate="one_to_one")
    .merge(icu_stay_count, on="subject_id", how="left", validate="one_to_one")
)
assert first_icu[["anchor_age", "anchor_year"]].notna().all().all()
first_icu["age_at_icu"] = (
    first_icu["anchor_age"] + first_icu["intime"].dt.year - first_icu["anchor_year"]
)
df_step1 = first_icu.loc[first_icu.age_at_icu.ge(18)].copy().reset_index(drop=True)
df = df_step1

assert df_step1.subject_id.is_unique
assert df_step1.stay_id.is_unique
assert df_step1.age_at_icu.ge(18).all()
assert len(df_step1) == 65_366, f"STEP1の患者数が65,366と異なります: {len(df_step1):,}"

step1_counts = pd.Series({
    "全ICU入室記録（件）": len(icustays),
    "患者ごとの初回ICU入室（人）": len(first_icu),
    "18歳以上の初回ICU入室（人）": len(df_step1),
}, name="N").to_frame()
display(step1_counts)
print(f"STEP1: N = {len(df_step1):,}人")
display(df_step1.head())

,N
全ICU入室記録（件）,94458
患者ごとの初回ICU入室（人）,65366
18歳以上の初回ICU入室（人）,65366


STEP1: N = 65,366人


,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,anchor_age,anchor_year,anchor_year_group,dod,icu_stay_count,age_at_icu
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266,F,52,2180,2014 - 2016,2180-09-09,1,52
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252,F,86,2150,2008 - 2010,2152-01-30,1,86
2,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535,F,73,2186,2008 - 2010,2193-08-26,1,76
3,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032,F,55,2157,2011 - 2013,NaN,2,55
4,10001725,25563031,31205490,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2110-04-11 15:52:22,2110-04-12 23:59:56,1.338588,F,46,2110,2011 - 2013,NaN,1,46


In [43]:
diagnoses_icd = pd.read_csv(
    DATA_DIR / "hosp/diagnoses_icd.csv.gz",
    usecols=["subject_id", "hadm_id", "seq_num", "icd_code", "icd_version"],
    dtype={"icd_code": "string"},
)

# 対象患者の初回ICUと同じ入院の全診断を取り出す。
same_admission_diagnoses = diagnoses_icd.loc[
    diagnoses_icd["hadm_id"].isin(df_step1["hadm_id"])
].copy()

# 初回ICUと同じ病院入院の全診断を結合する（患者1人につき複数行）。
df_step1_with_diagnosis = df_step1.merge(
    same_admission_diagnoses,
    on=["subject_id", "hadm_id"],
    how="left",
    validate="one_to_many",
)
df = df_step1_with_diagnosis

assert df_step1_with_diagnosis["subject_id"].nunique() == len(df_step1)
patients_with_icd = df_step1_with_diagnosis.loc[
    df_step1_with_diagnosis["icd_code"].notna(), "subject_id"
].nunique()
print(f"結合後の患者数: {df_step1_with_diagnosis['subject_id'].nunique():,}人")
print(f"結合後の行数: {len(df_step1_with_diagnosis):,}行")
print(f"ICDコードあり: {patients_with_icd:,}人")
print(f"ICDコードなし: {len(df_step1) - patients_with_icd:,}人")
display(df_step1_with_diagnosis.head())


結合後の患者数: 65,366人
結合後の行数: 1,110,887行
ICDコードあり: 65,349人
ICDコードなし: 17人


,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,anchor_age,anchor_year,anchor_year_group,dod,icu_stay_count,age_at_icu,seq_num,icd_code,icd_version
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266,F,52,2180,2014 - 2016,2180-09-09,1,52,1.0,45829,9.0
1,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266,F,52,2180,2014 - 2016,2180-09-09,1,52,2.0,07044,9.0
2,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266,F,52,2180,2014 - 2016,2180-09-09,1,52,3.0,7994,9.0
3,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266,F,52,2180,2014 - 2016,2180-09-09,1,52,4.0,2761,9.0
4,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266,F,52,2180,2014 - 2016,2180-09-09,1,52,5.0,78959,9.0


In [44]:
# 候補A：指定のICDコードで、同一入院の全診断を対象に抽出する。
# 元のdf_step1_with_diagnosisと他の候補データは保持する。
def candidate_a_icd_masks(diagnosis_df):
    code = (
        diagnosis_df["icd_code"].astype("string")
        .str.replace(".", "", regex=False).str.strip().str.upper()
    )
    version = diagnosis_df["icd_version"]
    icd9_mask = version.eq(9) & (
        code.str.startswith(("433", "434"), na=False)
        | code.isin(["4375", "4376"])
    )
    icd10_mask = version.eq(10) & (
        code.str.startswith(
            ("I630", "I631", "I632", "I633", "I634", "I635", "I636", "I637", "I66"),
            na=False,
        )
    )
    return icd9_mask, icd10_mask


# 後続の既存候補で使うマスクは、同一入院の全診断データに対応させる。
candidate_a_icd9_mask, candidate_a_icd10_mask = candidate_a_icd_masks(
    df_step1_with_diagnosis
)

candidate_a_same_admission = diagnoses_icd.loc[
    diagnoses_icd["hadm_id"].isin(df_step1["hadm_id"])
].copy()
candidate_a_all_icd9_mask, candidate_a_all_icd10_mask = candidate_a_icd_masks(
    candidate_a_same_admission
)
candidate_a_matching_diagnoses = candidate_a_same_admission.loc[
    candidate_a_all_icd9_mask | candidate_a_all_icd10_mask
].copy()

# 複数の診断が該当する場合は、該当診断のうちseq_numが最小の行を代表として保持する。
candidate_a_matching_diagnoses = (
    candidate_a_matching_diagnoses.sort_values("seq_num", kind="stable")
    .drop_duplicates(["subject_id", "hadm_id"])
)
df_candidate_a = df_step1.merge(
    candidate_a_matching_diagnoses,
    on=["subject_id", "hadm_id"],
    how="inner",
    validate="one_to_one",
)

assert df_candidate_a["subject_id"].is_unique
print(f"抽出前: {len(df_step1):,}人")
print(f"候補A: {len(df_candidate_a):,}人")
display(df_candidate_a.groupby("icd_version").size().rename("n_patients").to_frame())
display(df_candidate_a.head())


抽出前: 65,366人
候補A: 3,892人


,n_patients
icd_version,
9,2019
10,1873


,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,anchor_age,anchor_year,anchor_year_group,dod,icu_stay_count,age_at_icu,seq_num,icd_code,icd_version
0,10004733,27411876,39635619,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2174-12-04 11:28:24,2174-12-12 20:03:01,8.357373,M,51,2174,2014 - 2016,NaN,1,51,2,43491,9
1,10008100,29402054,35997892,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2181-12-10 19:41:59,2181-12-11 15:07:56,0.809688,F,86,2181,2008 - 2010,NaN,1,86,10,43310,9
2,10010888,20162667,33318955,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2174-01-09 00:21:00,2174-01-18 00:48:41,9.019225,M,43,2174,2020 - 2022,NaN,1,43,9,I63219,10
3,10020349,21891870,33383880,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2151-06-12 14:32:00,2151-06-21 11:35:55,8.877720,F,36,2151,2020 - 2022,NaN,1,36,2,I6302,10
4,10024982,25154057,37919901,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2203-09-20 18:27:39,2203-10-11 22:39:25,21.174838,M,78,2196,2008 - 2010,2203-10-11,1,85,9,I6310,10


In [45]:
# pmed.1004539.s006.pdf のSupplementary Table 2に記載のICDコード。
# 既存の成人・初回ICU・同一入院の全診断を対象とする。
# ICD-9: 433.X1, 434.X1, 437.1, 437.9
# ICD-10: I63.X, I67.81, I67.89, I67.9
supplementary_code = (
    df_step1_with_diagnosis["icd_code"].astype("string")
    .str.replace(".", "", regex=False).str.strip().str.upper()
)
supplementary_version = df_step1_with_diagnosis["icd_version"]

supplementary_icd9_mask = supplementary_version.eq(9) & (
    supplementary_code.str.fullmatch(r"(?:433|434)[0-9]1", na=False)
    | supplementary_code.isin(["4371", "4379"])
)
supplementary_icd10_mask = supplementary_version.eq(10) & (
    supplementary_code.str.startswith("I63", na=False)
    | supplementary_code.isin(["I6781", "I6789", "I679"])
)

df_supplementary_table2 = df_step1_with_diagnosis.loc[
    supplementary_icd9_mask | supplementary_icd10_mask
].copy()
# 条件に該当する診断のうちseq_numが最小の行を患者ごとの代表として保持する。
df_supplementary_table2 = (
    df_supplementary_table2.sort_values("seq_num", kind="stable")
    .drop_duplicates(["subject_id", "hadm_id"])
)

assert df_supplementary_table2["subject_id"].is_unique
print(f"抽出前: {df_step1_with_diagnosis['subject_id'].nunique():,}人")
print(f"付票Supplementary Table 2のコードに該当: {len(df_supplementary_table2):,}人")
display(df_supplementary_table2.groupby("icd_version").size().rename("n_patients").to_frame())
display(df_supplementary_table2.head())


抽出前: 65,366人
付票Supplementary Table 2のコードに該当: 3,941人


,n_patients
icd_version,
9.0,1439
10.0,2502


,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,anchor_age,anchor_year,anchor_year_group,dod,icu_stay_count,age_at_icu,seq_num,icd_code,icd_version
3137,10023708,21451830,38348907,Neuro Intermediate,Neuro Intermediate,2151-03-09 10:52:39,2151-03-13 14:42:21,4.159514,F,82,2143,2008 - 2010,NaN,1,90,1.0,I638,10.0
4083,10034171,28485870,34728076,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2119-04-25 16:43:04,2119-04-26 23:44:01,1.292326,F,76,2119,2017 - 2019,2119-04-26,1,76,1.0,I63131,10.0
4701,10039396,27932215,38451932,Neuro Intermediate,Neuro Intermediate,2142-02-01 02:49:52,2142-02-03 12:22:07,2.397396,M,91,2142,2017 - 2019,NaN,1,91,1.0,I63412,10.0
5030,10042682,27912536,30679441,Trauma SICU (TSICU),Trauma SICU (TSICU),2141-02-23 18:01:16,2141-02-24 22:03:15,1.168044,M,53,2141,2020 - 2022,NaN,1,53,1.0,I63419,10.0
6094,10052047,23949557,34050312,Neuro Intermediate,Neuro Intermediate,2201-02-07 17:33:00,2201-02-15 17:20:11,7.991100,M,80,2191,2011 - 2013,NaN,1,90,1.0,I63233,10.0


In [46]:
# 候補AとSupplementary Table 2のICD条件のOR（和集合）で抽出する。
# 成人・初回ICU・同一入院の全診断という条件は共通。
# 元のデータと既存の候補データは保持する。
candidate_a_or_supplementary_mask = (
    candidate_a_icd9_mask | candidate_a_icd10_mask
    | supplementary_icd9_mask | supplementary_icd10_mask
)
df_candidate_a_or_supplementary = df_step1_with_diagnosis.loc[
    candidate_a_or_supplementary_mask
].copy()
# 条件に該当する診断のうちseq_numが最小の行を患者ごとの代表として保持する。
df_candidate_a_or_supplementary = (
    df_candidate_a_or_supplementary.sort_values("seq_num", kind="stable")
    .drop_duplicates(["subject_id", "hadm_id"])
)

assert df_candidate_a_or_supplementary["subject_id"].is_unique
print(f"抽出前: {df_step1_with_diagnosis['subject_id'].nunique():,}人")
print(f"候補A: {len(df_candidate_a):,}人")
print(f"Supplementary Table 2: {len(df_supplementary_table2):,}人")
print(f"2つの条件のOR: {len(df_candidate_a_or_supplementary):,}人")
display(
    df_candidate_a_or_supplementary.groupby("icd_version")
    .size().rename("n_patients").to_frame()
)
display(df_candidate_a_or_supplementary.head())


抽出前: 65,366人
候補A: 3,892人
Supplementary Table 2: 3,941人
2つの条件のOR: 4,661人


,n_patients
icd_version,
9.0,2118
10.0,2543


,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,anchor_age,anchor_year,anchor_year_group,dod,icu_stay_count,age_at_icu,seq_num,icd_code,icd_version
3137,10023708,21451830,38348907,Neuro Intermediate,Neuro Intermediate,2151-03-09 10:52:39,2151-03-13 14:42:21,4.159514,F,82,2143,2008 - 2010,NaN,1,90,1.0,I638,10.0
4083,10034171,28485870,34728076,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2119-04-25 16:43:04,2119-04-26 23:44:01,1.292326,F,76,2119,2017 - 2019,2119-04-26,1,76,1.0,I63131,10.0
4701,10039396,27932215,38451932,Neuro Intermediate,Neuro Intermediate,2142-02-01 02:49:52,2142-02-03 12:22:07,2.397396,M,91,2142,2017 - 2019,NaN,1,91,1.0,I63412,10.0
5030,10042682,27912536,30679441,Trauma SICU (TSICU),Trauma SICU (TSICU),2141-02-23 18:01:16,2141-02-24 22:03:15,1.168044,M,53,2141,2020 - 2022,NaN,1,53,1.0,I63419,10.0
6094,10052047,23949557,34050312,Neuro Intermediate,Neuro Intermediate,2201-02-07 17:33:00,2201-02-15 17:20:11,7.991100,M,80,2191,2011 - 2013,NaN,1,90,1.0,I63233,10.0


## 検証1：初回ICUコホートと生の全診断を直接結合する

既存の診断抽出結果や患者ごとの代表診断を使わず、`df_step1`の`subject_id`・`hadm_id`を生の`diagnoses_icd.csv.gz`全行と結合する。成人判定と患者ごとの初回ICU選択はSTEP1と共通にする。診断条件を適用する前には重複排除しない。


In [47]:
audit_cohort_keys = df_step1[["subject_id", "hadm_id"]].copy()
assert audit_cohort_keys["subject_id"].is_unique
assert not audit_cohort_keys.duplicated(["subject_id", "hadm_id"]).any()
assert len(audit_cohort_keys) == 65_366

# 以前の診断抽出結果に依存しないよう、生の全行を読み直す。
audit_raw_diagnoses = pd.read_csv(
    DATA_DIR / "hosp/diagnoses_icd.csv.gz",
    usecols=["subject_id", "hadm_id", "seq_num", "icd_code", "icd_version"],
    dtype={"icd_code": "string"},
)
audit_raw_row_count = len(audit_raw_diagnoses)
audit_first_icu_diagnoses = audit_cohort_keys.merge(
    audit_raw_diagnoses,
    on=["subject_id", "hadm_id"],
    how="inner",
    validate="one_to_many",
)
del audit_raw_diagnoses
assert audit_first_icu_diagnoses["seq_num"].notna().all()
assert audit_first_icu_diagnoses["icd_version"].isin([9, 10]).all()
assert not audit_first_icu_diagnoses.duplicated(
    ["subject_id", "hadm_id", "seq_num"]
).any()
audit_code = (
    audit_first_icu_diagnoses["icd_code"].astype("string")
    .str.replace(".", "", regex=False).str.strip().str.upper()
)
audit_version = audit_first_icu_diagnoses["icd_version"]
audit_seq1 = audit_first_icu_diagnoses["seq_num"].eq(1)
audit_join_summary = pd.Series({
    "成人の初回ICU患者数": len(audit_cohort_keys),
    "生の全診断行数": audit_raw_row_count,
    "同一入院に結合された診断行数": len(audit_first_icu_diagnoses),
    "同一入院の診断あり患者数": audit_first_icu_diagnoses["subject_id"].nunique(),
    "同一入院の診断なし患者数": len(audit_cohort_keys)
        - audit_first_icu_diagnoses["subject_id"].nunique(),
}, name="N").to_frame()
display(audit_join_summary)


,N
成人の初回ICU患者数,65366
生の全診断行数,6364488
同一入院に結合された診断行数,1110870
同一入院の診断あり患者数,65349
同一入院の診断なし患者数,17


## 検証2：A1・A2・B1・B2を比較する

- A：既存の候補A。ICD-9は`433* / 434* / 4375 / 4376`、ICD-10は`I630*〜I637* / I66*`。
- B：ICD-9は`433* / 434* / 436 / 4370 / 4371`、ICD-10は`I63* / I65* / I66*`。
- A1・B1は診断順位の制限なし、A2・B2は検証用に`seq_num = 1`を適用する。既存の抽出条件は変更しない。

小数点を除去してICDの版も照合する。`*`は前方一致、`436 / 4370 / 4371 / 4375 / 4376`は完全一致。患者数は`subject_id.nunique()`で数え、診断行数とは区別する。比較対象の5,279人はユーザー提示の値であり、人数の近さだけでは論文の実装を特定できない。


In [48]:
audit_a_icd9, audit_a_icd10 = candidate_a_icd_masks(audit_first_icu_diagnoses)
audit_a_mask = audit_a_icd9 | audit_a_icd10
audit_b_mask = (
    audit_version.eq(9) & (
        audit_code.str.startswith(("433", "434"), na=False)
        | audit_code.isin(["436", "4370", "4371"])
    )
) | (
    audit_version.eq(10)
    & audit_code.str.startswith(("I63", "I65", "I66"), na=False)
)
audit_condition_masks = {
    "A1": audit_a_mask,
    "A2": audit_a_mask & audit_seq1,
    "B1": audit_b_mask,
    "B2": audit_b_mask & audit_seq1,
}
audit_patient_sets = {}
audit_count_rows = []
for label, mask in audit_condition_masks.items():
    matching = audit_first_icu_diagnoses.loc[mask]
    patient_ids = set(matching["subject_id"])
    audit_patient_sets[label] = patient_ids
    audit_count_rows.append({
        "条件": label,
        "コード集合": label[0],
        "診断順位": "制限なし" if label.endswith("1") else "seq_num = 1",
        "患者数": len(patient_ids),
        "病院入院数": matching["hadm_id"].nunique(),
        "診断行数": len(matching),
        "5279人との差": len(patient_ids) - 5_279,
    })
audit_four_conditions = pd.DataFrame(audit_count_rows).set_index("条件")
assert audit_patient_sets["A2"] <= audit_patient_sets["A1"]
assert audit_patient_sets["B2"] <= audit_patient_sets["B1"]
assert (audit_four_conditions["患者数"] == audit_four_conditions["病院入院数"]).all()
display(audit_four_conditions)
audit_rank_effect = pd.DataFrame([
    {"コード集合": label, "seq_num制限解除で増える患者数":
        len(audit_patient_sets[label + "1"] - audit_patient_sets[label + "2"])}
    for label in ("A", "B")
]).set_index("コード集合")
display(audit_rank_effect)


,コード集合,診断順位,患者数,病院入院数,診断行数,5279人との差
条件,,,,,,
A1,A,制限なし,3892,3892,4634,-1387
A2,A,seq_num = 1,2124,2124,2124,-3155
B1,B,制限なし,5263,5263,6378,-16
B2,B,seq_num = 1,2457,2457,2457,-2822


,seq_num制限解除で増える患者数
コード集合,
A,1768
B,2806


## 検証3：指定コードを候補Aに一つずつ追加する

A1は3,892人で、比較対象の5,279人より1,387人少なかった。そこで、ICD-9 `436 / 4370 / 4371`、ICD-10 `I65* / I638* / I639*`をそれぞれAに追加する。

各行は同じAを基準とする独立した追加。コード該当患者のうち、既にAに含まれる患者は新規患者に数えない。A1とA2を別々に検証する。


In [49]:
audit_extra_masks = {
    "436": audit_version.eq(9) & audit_code.eq("436"),
    "4370": audit_version.eq(9) & audit_code.eq("4370"),
    "4371": audit_version.eq(9) & audit_code.eq("4371"),
    "I65*": audit_version.eq(10) & audit_code.str.startswith("I65", na=False),
    "I638*": audit_version.eq(10) & audit_code.str.startswith("I638", na=False),
    "I639*": audit_version.eq(10) & audit_code.str.startswith("I639", na=False),
}
audit_individual_rows = []
for label, extra_mask in audit_extra_masks.items():
    extra_ids = set(audit_first_icu_diagnoses.loc[extra_mask, "subject_id"])
    extra_seq1_ids = set(audit_first_icu_diagnoses.loc[extra_mask & audit_seq1, "subject_id"])
    a1_new = extra_ids - audit_patient_sets["A1"]
    a2_new = extra_seq1_ids - audit_patient_sets["A2"]
    audit_individual_rows.append({
        "追加コード": label,
        "コード該当患者数（全順位）": len(extra_ids),
        "A1との重複患者数": len(extra_ids & audit_patient_sets["A1"]),
        "A1からの新規患者数": len(a1_new),
        "A1＋コードの患者数": len(audit_patient_sets["A1"] | extra_ids),
        "A2からの新規患者数": len(a2_new),
        "A2＋コードの患者数": len(audit_patient_sets["A2"] | extra_seq1_ids),
    })
audit_individual_additions = pd.DataFrame(audit_individual_rows).set_index("追加コード")
display(audit_individual_additions)


,コード該当患者数（全順位）,A1との重複患者数,A1からの新規患者数,A1＋コードの患者数,A2からの新規患者数,A2＋コードの患者数
追加コード,,,,,,
436,0,0,0,3892,0,2124
4370,189,48,141,4033,2,2126
4371,10,7,3,3895,0,2124
I65*,869,210,659,4551,91,2215
I638*,275,24,251,4143,128,2252
I639*,394,18,376,4268,120,2244


## 検証4：指定順にコードを追加し、Bとの差を確認する

`436 → 4370 → 4371 → I65* → I638* → I639*`の順にAへ追加する。各段階の新規患者数は、その直前の患者集合との差。複数コードに該当する患者を重複して加算しない。

A＋全追加コードは`4375 / 4376`を保持するため、Bとは異なる。患者集合の和集合・共通部分・差集合も確認する。


In [50]:
audit_cumulative_a1 = set(audit_patient_sets["A1"])
audit_cumulative_a2 = set(audit_patient_sets["A2"])
audit_expanded_a_mask = audit_a_mask.copy()
audit_cumulative_rows = [{
    "段階": "候補A（追加前）", "全順位の患者数": len(audit_cumulative_a1),
    "直前からの増加（全順位）": 0, "seq_num=1の患者数": len(audit_cumulative_a2),
    "直前からの増加（seq_num=1）": 0,
    "5279人との差（全順位）": len(audit_cumulative_a1) - 5_279,
}]
for label, extra_mask in audit_extra_masks.items():
    before_a1, before_a2 = len(audit_cumulative_a1), len(audit_cumulative_a2)
    audit_cumulative_a1.update(audit_first_icu_diagnoses.loc[extra_mask, "subject_id"])
    audit_cumulative_a2.update(audit_first_icu_diagnoses.loc[extra_mask & audit_seq1, "subject_id"])
    audit_expanded_a_mask |= extra_mask
    audit_cumulative_rows.append({
        "段階": f"さらに{label}を追加",
        "全順位の患者数": len(audit_cumulative_a1),
        "直前からの増加（全順位）": len(audit_cumulative_a1) - before_a1,
        "seq_num=1の患者数": len(audit_cumulative_a2),
        "直前からの増加（seq_num=1）": len(audit_cumulative_a2) - before_a2,
        "5279人との差（全順位）": len(audit_cumulative_a1) - 5_279,
    })
audit_cumulative_additions = pd.DataFrame(audit_cumulative_rows).set_index("段階")
display(audit_cumulative_additions)

# 実データ上で、A＋6コードとA・BのORの診断マスクが一致するかも確認する。
assert audit_expanded_a_mask.equals(audit_a_mask | audit_b_mask)
assert audit_cumulative_a1 == audit_patient_sets["A1"] | audit_patient_sets["B1"]
assert audit_cumulative_a2 == audit_patient_sets["A2"] | audit_patient_sets["B2"]
audit_set_difference = pd.DataFrame([
    {
        "診断順位": rank,
        "AとBの共通患者数": len(audit_patient_sets[a] & audit_patient_sets[b]),
        "Aのみの患者数": len(audit_patient_sets[a] - audit_patient_sets[b]),
        "Bのみの患者数": len(audit_patient_sets[b] - audit_patient_sets[a]),
        "A・BのOR患者数": len(audit_patient_sets[a] | audit_patient_sets[b]),
    }
    for rank, a, b in (("制限なし", "A1", "B1"), ("seq_num = 1", "A2", "B2"))
]).set_index("診断順位")
display(audit_set_difference)
audit_a_only_rows = []
for rank, a, b in (("制限なし", "A1", "B1"), ("seq_num = 1", "A2", "B2")):
    a_only_ids = audit_patient_sets[a] - audit_patient_sets[b]
    for label in ("4375", "4376"):
        mask = audit_version.eq(9) & audit_code.eq(label)
        if rank == "seq_num = 1":
            mask &= audit_seq1
        matching_ids = set(audit_first_icu_diagnoses.loc[mask, "subject_id"])
        audit_a_only_rows.append({
            "診断順位": rank, "コード": label,
            "Aのみ患者のうち該当する患者数": len(a_only_ids & matching_ids),
        })
audit_a_only_codes = pd.DataFrame(audit_a_only_rows).set_index(["診断順位", "コード"])
display(audit_a_only_codes)


,全順位の患者数,直前からの増加（全順位）,seq_num=1の患者数,直前からの増加（seq_num=1）,5279人との差（全順位）
段階,,,,,
候補A（追加前）,3892,0,2124,0,-1387
さらに436を追加,3892,0,2124,0,-1387
さらに4370を追加,4033,141,2126,2,-1246
さらに4371を追加,4036,3,2126,0,-1243
さらにI65*を追加,4695,659,2217,91,-584
さらにI638*を追加,4927,232,2345,128,-352
さらにI639*を追加,5279,352,2465,120,0


,AとBの共通患者数,Aのみの患者数,Bのみの患者数,A・BのOR患者数
診断順位,,,,
制限なし,3876,16,1387,5279
seq_num = 1,2116,8,341,2465


Aのみ患者のうち該当する患者数
診断順位        コード                  
制限なし        4375               10
            4376                6
seq_num = 1 4375                7
            4376                1

## 検証5：最終コード集合を明示し、別の集計方法で照合する

全診断順位でAに6コードを追加すると5,279人になった。B1の5,263人との差は、Aにだけ含まれる`4375`の10人と`4376`の6人。最終集合は以下のとおり。

- ICD-9：`433* / 434* / 436 / 4370 / 4371 / 4375 / 4376`
- ICD-10：`I63* / I65* / I66*`
- 成人の初回ICUと同じ`subject_id`・`hadm_id`の診断に限定し、診断順位の制限は設けない。

この集合を直接指定して、患者ごとの診断該当フラグを`groupby().any()`で集計する。集合演算による前段の人数と照合し、全順位の候補を`df_candidate_a_expanded`として患者1人1行にまとめる。代表診断は該当診断のうち`seq_num`が最小の行とするが、患者の採否には順位を使用しない。

**結果：全診断順位では5,279人、同じコード集合の`seq_num = 1`では2,465人。** 手元の条件では提示された第2段階の人数を再現できた。コード集合と診断順位の両方が人数に影響するため、これだけで著者の実装や“primary diagnosis”の意味、Methodsとの不一致までは確定できない。既存の候補A・Bとは別の追加検証結果として扱う。


In [51]:
audit_final_mask = (
    audit_version.eq(9) & (
        audit_code.str.startswith(("433", "434"), na=False)
        | audit_code.isin(["436", "4370", "4371", "4375", "4376"])
    )
) | (
    audit_version.eq(10)
    & audit_code.str.startswith(("I63", "I65", "I66"), na=False)
)
assert audit_final_mask.equals(audit_expanded_a_mask)
audit_patient_flags = pd.DataFrame({
    "subject_id": audit_first_icu_diagnoses["subject_id"],
    "全診断順位": audit_final_mask,
    "seq_num = 1": audit_final_mask & audit_seq1,
}).groupby("subject_id")[["全診断順位", "seq_num = 1"]].any()
audit_final_counts = audit_patient_flags.sum().astype("int64")
assert int(audit_final_counts["全診断順位"]) == len(audit_cumulative_a1)
assert int(audit_final_counts["seq_num = 1"]) == len(audit_cumulative_a2)
# 診断行の重複排除はICD条件を適用した後に行う。
audit_expanded_representatives = (
    audit_first_icu_diagnoses.loc[audit_final_mask]
    .sort_values("seq_num", kind="stable")
    .drop_duplicates(["subject_id", "hadm_id"])
)
df_candidate_a_expanded = df_step1.merge(
    audit_expanded_representatives,
    on=["subject_id", "hadm_id"],
    how="inner",
    validate="one_to_one",
)
assert df_candidate_a_expanded["subject_id"].is_unique
assert set(df_candidate_a_expanded["subject_id"]) == audit_cumulative_a1
audit_final_summary = audit_final_counts.rename("患者数").to_frame()
audit_final_summary["5279人との差"] = audit_final_summary["患者数"] - 5_279
display(audit_final_summary)


,患者数,5279人との差
全診断順位,5279,0
seq_num = 1,2465,-2814


## ICU滞在24時間以上を抽出する

最終コード集合に該当した`df_candidate_a_expanded`から、初回ICUの滞在時間が24時間以上の患者を`df_icu_24h`に抽出する。滞在時間は`outtime - intime`から算出し、ちょうど24時間も含める。入退室時刻が欠測している患者は除外する。


In [52]:
# 最終コード集合に該当する患者に、初回ICU滞在24時間以上の条件を適用する。
icu_duration = (
    df_candidate_a_expanded["outtime"] - df_candidate_a_expanded["intime"]
)
icu_24h_mask = icu_duration.ge(pd.Timedelta(hours=24))
df_icu_24h = df_candidate_a_expanded.loc[icu_24h_mask].copy()
df_icu_24h["icu_los_hours"] = (
    icu_duration.loc[icu_24h_mask].dt.total_seconds() / 3600
)

assert df_icu_24h["subject_id"].is_unique
assert df_icu_24h["icu_los_hours"].ge(24).all()
icu_24h_counts = pd.Series({
    "抽出前（人）": len(df_candidate_a_expanded),
    "ICU滞在24時間以上（人）": len(df_icu_24h),
    "ICU滞在24時間未満（人）": int(icu_duration.lt(pd.Timedelta(hours=24)).sum()),
    "入退室時刻が欠測（人）": int(icu_duration.isna().sum()),
}, name="N").to_frame()
display(icu_24h_counts)
print(f"ICU滞在24時間以上: N = {len(df_icu_24h):,}人")
display(df_icu_24h.head())


,N
抽出前（人）,5279
ICU滞在24時間以上（人）,4632
ICU滞在24時間未満（人）,647
入退室時刻が欠測（人）,0


ICU滞在24時間以上: N = 4,632人


,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,anchor_age,anchor_year,anchor_year_group,dod,icu_stay_count,age_at_icu,seq_num,icd_code,icd_version,icu_los_hours
0,10004733,27411876,39635619,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2174-12-04 11:28:24,2174-12-12 20:03:01,8.357373,M,51,2174,2014 - 2016,NaN,1,51,2,43491,9,200.576944
3,10010888,20162667,33318955,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2174-01-09 00:21:00,2174-01-18 00:48:41,9.019225,M,43,2174,2020 - 2022,NaN,1,43,9,I63219,10,216.461389
4,10013643,22009484,33072499,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2200-10-01 08:42:08,2200-10-05 18:16:16,4.398704,F,79,2195,2008 - 2010,NaN,1,84,23,4370,9,105.568889
5,10015186,23922627,32286205,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2123-11-05 21:57:57,2123-11-15 13:14:29,9.636481,F,78,2123,2020 - 2022,NaN,1,78,20,I6529,10,231.275556
6,10020349,21891870,33383880,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2151-06-12 14:32:00,2151-06-21 11:35:55,8.877720,F,36,2151,2020 - 2022,NaN,1,36,2,I6302,10,213.065278


## Time-dependent Cox用の共変量を抽出する

`df_icu_24h`を対象に、主解析のスタチン・年齢・性別・人種と、サブグループ解析の6併存疾患（糖尿病、高血圧、腎疾患、心不全、脳血管疾患、認知症）を患者1人1行の`df_cox_covariates`にまとめる。欠測による除外や補完はこのセルでは行わない。

論文のtime-dependent Coxは、入室時のスタチン曝露と`log(t)`の交互作用で係数の時間変化を表す。処方開始後に曝露を切り替えるモデルとは定義が異なる。`statin × log(t)`はCox推定時のリスク集合の時刻で評価する必要があるため、患者の最終観測時間を使った固定列は作らない。主解析の列を`cox_main_covariate_cols`、サブグループの列を`cox_subgroup_cols`に保持する。Table 1の検査値・重症度スコアは本文で主解析の調整変数として挙げられていないため、このセルの対象には含めない。

- 年齢は既存の`age_at_icu`、性別は`gender`から取得する。人種は同じ患者・入院の`admissions.race`からWhite / Black / Otherにまとめ、UNKNOWN・UNABLE TO OBTAIN・PATIENT DECLINED TO ANSWERなどは欠測として保持する。元の値も`race_raw`に残す。
- 論文のスタチンは、自宅の服薬リストまたは入院時の薬剤照合で確認したベースライン曝露。構造化`prescriptions`だけではこれを確定できない。ここでの`statin`は**同じ入院でICU入室時に有効な院内処方があるかという代理指標**とし、確認用に`statin_prescription_rows`を残す。対象は7成分と対応する商品名・配合剤名。`starttime <= intime < stoptime`を満たす処方を1、該当処方がない場合を0とする。ただし処方記録自体がない患者や、有効な処方の有無を時刻欠測で判定できない患者はNAとする。0は入室前の服薬なしを意味しない。服薬歴が未確認であることを`statin_home_use`（全例NA）で明示する。
- 入室後のスタチン開始はベースラインの1に含めず、別列`statin_started_after_icu`に記録する。処方と実際の投与も区別する。
- 併存疾患は同じ入院の**全診断**から集計し、脳卒中の代表診断1件だけは使わない。高血圧はICD-9 401–405 / ICD-10 I10–I15、糖尿病はICD-9 250 / ICD-10 E10–E14、残り4疾患はMIMIC公式Charlson定義に合わせる。これらは退院時診断に基づく記録フラグで、発症時刻や入室前からの既往を保証しない。診断がない患者はNAとする。

出典：[論文Methods](https://www.frontiersin.org/journals/aging-neuroscience/articles/10.3389/fnagi.2026.1866416/full)、[MIMIC公式Charlson SQL](https://github.com/MIT-LCP/mimic-code/blob/main/mimic-iv/concepts/comorbidity/charlson.sql)。


In [53]:
# 既存コホートを保持し、共変量用のDataFrameを別に作る。
cox_keys = ["subject_id", "hadm_id"]
cox_cohort = df_icu_24h[
    cox_keys + ["stay_id", "intime", "outtime", "age_at_icu", "gender"]
].copy()
assert cox_cohort["subject_id"].is_unique
assert cox_cohort["intime"].notna().all()
cox_hadm_ids = set(cox_cohort["hadm_id"])

cox_admissions = pd.read_csv(
    DATA_DIR / "hosp/admissions.csv.gz",
    usecols=cox_keys + ["race"], dtype={"race": "string"},
)
cox_admissions = cox_admissions.loc[cox_admissions["hadm_id"].isin(cox_hadm_ids)]
df_cox_covariates = cox_cohort.merge(
    cox_admissions.rename(columns={"race": "race_raw"}),
    on=cox_keys, how="left", validate="one_to_one",
)
df_cox_covariates["age"] = df_cox_covariates["age_at_icu"]
df_cox_covariates["male"] = df_cox_covariates["gender"].map({"M": 1, "F": 0}).astype("Int64")
cox_race = df_cox_covariates["race_raw"].str.strip().str.upper()
cox_race_unknown = cox_race.isna() | cox_race.isin([
    "", "UNKNOWN", "UNABLE TO OBTAIN", "PATIENT DECLINED TO ANSWER",
])
df_cox_covariates["race"] = pd.Series("Other", index=df_cox_covariates.index, dtype="string")
df_cox_covariates.loc[cox_race.str.startswith("WHITE", na=False), "race"] = "White"
df_cox_covariates.loc[cox_race.str.startswith("BLACK", na=False), "race"] = "Black"
df_cox_covariates.loc[cox_race_unknown, "race"] = pd.NA
for group in ("Black", "Other"):
    col = "race_" + group.lower()
    df_cox_covariates[col] = df_cox_covariates["race"].eq(group).astype("Int64")

# 全診断を使う。診断順位や脳卒中コードによる絞り込みはしない。
cox_diagnoses = diagnoses_icd.loc[diagnoses_icd["hadm_id"].isin(cox_hadm_ids)].merge(
    cox_cohort[cox_keys], on=cox_keys, how="inner", validate="many_to_one",
)
cox_code = (cox_diagnoses["icd_code"].astype("string")
            .str.replace(".", "", regex=False).str.strip().str.upper())
cox_v9 = cox_diagnoses["icd_version"].eq(9)
cox_v10 = cox_diagnoses["icd_version"].eq(10)
cox_c3, cox_c4, cox_c5 = cox_code.str[:3], cox_code.str[:4], cox_code.str[:5]
cox_icd_masks = {
    "diabetes": (cox_v9 & cox_c3.eq("250")) | (cox_v10 & cox_c3.between("E10", "E14")),
    "hypertension": (cox_v9 & cox_c3.between("401", "405")) | (cox_v10 & cox_c3.between("I10", "I15")),
    "kidney_disease": (
        cox_v9 & (
            cox_c3.isin(["582", "585", "586", "V56"])
            | cox_c4.isin(["5880", "V420", "V451"])
            | cox_c4.between("5830", "5837")
            | cox_c5.isin(["40301", "40311", "40391", "40402", "40403",
                            "40412", "40413", "40492", "40493"])
        )
    ) | (
        cox_v10 & (
            cox_c3.isin(["N18", "N19"])
            | cox_c4.isin(["I120", "I131", "N032", "N033", "N034", "N035", "N036", "N037",
                            "N052", "N053", "N054", "N055", "N056", "N057", "N250",
                            "Z490", "Z491", "Z492", "Z940", "Z992"])
        )
    ),
    "heart_failure": (
        cox_v9 & (
            cox_c3.eq("428") | cox_c4.between("4254", "4259")
            | cox_c5.isin(["39891", "40201", "40211", "40291", "40401", "40403",
                            "40411", "40413", "40491", "40493"])
        )
    ) | (
        cox_v10 & (
            cox_c3.isin(["I43", "I50"])
            | cox_c4.isin(["I099", "I110", "I130", "I132", "I255", "I420",
                            "I425", "I426", "I427", "I428", "I429", "P290"])
        )
    ),
    "cerebrovascular_disease": (
        cox_v9 & (cox_c3.between("430", "438") | cox_c5.eq("36234"))
    ) | (
        cox_v10 & (cox_c3.isin(["G45", "G46"]) | cox_c3.between("I60", "I69") | cox_c4.eq("H340"))
    ),
    "dementia": (
        cox_v9 & (cox_c3.eq("290") | cox_c4.isin(["2941", "3312"]))
    ) | (
        cox_v10 & (cox_c3.isin(["F00", "F01", "F02", "F03", "G30"]) | cox_c4.isin(["F051", "G311"]))
    ),
}
cox_subgroup_cols = list(cox_icd_masks)
cox_diagnosis_flags = cox_diagnoses[cox_keys].copy()
for name, mask in cox_icd_masks.items():
    cox_diagnosis_flags[name] = mask.fillna(False).astype("Int64")
cox_diagnosis_flags = cox_diagnosis_flags.groupby(cox_keys)[cox_subgroup_cols].max().reset_index()
df_cox_covariates = df_cox_covariates.merge(
    cox_diagnosis_flags, on=cox_keys, how="left", validate="one_to_one",
)

# 大きい処方CSVはチャンク読み込みし、対象入院の記録だけを保持する。
cox_statin_pattern = (
    r"\b(?:atorvastatin|rosuvastatin|simvastatin|pravastatin|lovastatin|fluvastatin|pitavastatin"
    r"|lipitor|crestor|zocor|pravachol|mevacor|altoprev|lescol|livalo|zypitamag"
    r"|caduet|vytorin|simcor|advicor)\b"
)
cox_rx_counts = pd.Series(0, index=pd.Index(sorted(cox_hadm_ids), name="hadm_id"), dtype="int64")
cox_statin_chunks = []
for chunk in pd.read_csv(
    DATA_DIR / "hosp/prescriptions.csv.gz",
    usecols=cox_keys + ["pharmacy_id", "drug", "starttime", "stoptime"],
    dtype={"drug": "string"}, chunksize=250_000,
):
    chunk = chunk.loc[chunk["hadm_id"].isin(cox_hadm_ids)].merge(
        cox_cohort[cox_keys], on=cox_keys, how="inner", validate="many_to_one",
    )
    cox_rx_counts = cox_rx_counts.add(chunk.groupby("hadm_id").size(), fill_value=0)
    matches = chunk["drug"].str.contains(cox_statin_pattern, case=False, regex=True, na=False)
    cox_statin_chunks.append(chunk.loc[matches].copy())
statin_prescription_rows = pd.concat(cox_statin_chunks, ignore_index=True).merge(
    cox_cohort[cox_keys + ["stay_id", "intime", "outtime"]],
    on=cox_keys, how="inner", validate="many_to_one",
)
for col in ("starttime", "stoptime"):
    statin_prescription_rows[col] = pd.to_datetime(statin_prescription_rows[col], errors="coerce")
cox_rx_start = statin_prescription_rows["starttime"]
cox_rx_stop = statin_prescription_rows["stoptime"]
cox_rx_icu = statin_prescription_rows["intime"]
statin_prescription_rows["active_at_icu"] = cox_rx_start.le(cox_rx_icu) & cox_rx_stop.gt(cox_rx_icu)
# 欠測時刻でも、既に終了した処方や入室後開始の処方は曖昧なベースラインに数えない。
statin_prescription_rows["baseline_timing_unknown"] = (
    (cox_rx_start.isna() & (cox_rx_stop.isna() | cox_rx_stop.gt(cox_rx_icu)))
    | (cox_rx_start.le(cox_rx_icu) & cox_rx_stop.isna())
)
statin_prescription_rows["started_after_icu"] = (
    cox_rx_start.gt(cox_rx_icu) & cox_rx_start.lt(statin_prescription_rows["outtime"])
)
cox_statin_flags = statin_prescription_rows.groupby(cox_keys)[
    ["active_at_icu", "baseline_timing_unknown", "started_after_icu"]
].any().reset_index()
df_cox_covariates = df_cox_covariates.merge(
    cox_statin_flags, on=cox_keys, how="left", validate="one_to_one",
)
df_cox_covariates["n_prescription_rows"] = df_cox_covariates["hadm_id"].map(cox_rx_counts).astype("Int64")
df_cox_covariates["statin"] = df_cox_covariates["active_at_icu"].eq(True).astype("Int64")
cox_uncertain_statin = (
    df_cox_covariates["n_prescription_rows"].eq(0)
    | (df_cox_covariates["baseline_timing_unknown"].eq(True) & df_cox_covariates["statin"].eq(0))
)
df_cox_covariates.loc[cox_uncertain_statin, "statin"] = pd.NA
df_cox_covariates["statin_home_use"] = pd.Series(pd.NA, index=df_cox_covariates.index, dtype="Int64")
df_cox_covariates["statin_started_after_icu"] = df_cox_covariates["started_after_icu"].eq(True).astype("Int64")
df_cox_covariates.loc[df_cox_covariates["n_prescription_rows"].eq(0), "statin_started_after_icu"] = pd.NA
df_cox_covariates = df_cox_covariates.drop(columns=["active_at_icu", "baseline_timing_unknown", "started_after_icu"])

# Whiteを参照群とする。30日モデルで人種を層別化する場合にはrace列を使う。
cox_main_covariate_cols = ["statin", "age", "male", "race_black", "race_other"]
cox_covariate_cols = cox_main_covariate_cols + cox_subgroup_cols
assert len(df_cox_covariates) == len(df_icu_24h)
assert df_cox_covariates["subject_id"].is_unique
assert set(df_cox_covariates["subject_id"]) == set(df_icu_24h["subject_id"])
for col in ["statin", "male", "race_black", "race_other"] + cox_subgroup_cols:
    assert df_cox_covariates[col].dropna().isin([0, 1]).all(), col
cox_covariate_missingness = pd.DataFrame({
    "欠測数": df_cox_covariates[cox_covariate_cols].isna().sum(),
    "欠測率": df_cox_covariates[cox_covariate_cols].isna().mean(),
})
print(f"共変量抽出: N = {len(df_cox_covariates):,}人（患者の追加除外なし）")
print("statinはICU入室時の有効な院内処方の代理指標。自宅での服薬歴は未確認です。")
display(df_cox_covariates["statin"].value_counts(dropna=False).rename("患者数").to_frame())
display(cox_covariate_missingness)
display(df_cox_covariates.head())


共変量抽出: N = 4,632人（患者の追加除外なし）
statinはICU入室時の有効な院内処方の代理指標。自宅での服薬歴は未確認です。


,患者数
statin,
0,3820
1,779
<NA>,33


,欠測数,欠測率
statin,33,0.007124
age,0,0.000000
male,0,0.000000
race_black,934,0.201641
race_other,934,0.201641
diabetes,0,0.000000
hypertension,0,0.000000
kidney_disease,0,0.000000
heart_failure,0,0.000000
cerebrovascular_disease,0,0.000000


,subject_id,hadm_id,stay_id,intime,outtime,age_at_icu,gender,race_raw,age,male,...,diabetes,hypertension,kidney_disease,heart_failure,cerebrovascular_disease,dementia,n_prescription_rows,statin,statin_home_use,statin_started_after_icu
0,10004733,27411876,39635619,2174-12-04 11:28:24,2174-12-12 20:03:01,51,M,UNKNOWN,51,1,...,0,1,1,0,1,0,184,0,<NA>,1
1,10010888,20162667,33318955,2174-01-09 00:21:00,2174-01-18 00:48:41,43,M,WHITE,43,1,...,1,0,0,0,1,0,210,0,<NA>,0
2,10013643,22009484,33072499,2200-10-01 08:42:08,2200-10-05 18:16:16,84,F,WHITE,84,0,...,0,0,0,1,1,1,110,1,<NA>,1
3,10015186,23922627,32286205,2123-11-05 21:57:57,2123-11-15 13:14:29,78,F,UNABLE TO OBTAIN,78,0,...,0,1,0,0,1,0,154,0,<NA>,0
4,10020349,21891870,33383880,2151-06-12 14:32:00,2151-06-21 11:35:55,36,F,HISPANIC/LATINO - GUATEMALAN,36,0,...,0,0,0,0,1,0,86,0,<NA>,0
